In [1]:
import sys
sys.path.append("..")

from src.data import load_config, load_raw, subsample, split, fit_scaler, apply_scaler, save_processed, load_processed

In [2]:
config = load_config("/Users/chris/Documents/higs-vision/config.yaml")
# Step 1: Load raw data
features, labels = load_raw(config)

# Step 2: Subsample
features, labels = subsample(features, labels, config["dataset"]["subsample_size"], config["seeds"]["data_split"])

# Step 3: Split
X_train, y_train, X_val, y_val, X_test, y_test = split(features, labels, config)

# Step 4 & 5: Fit and apply scaler
scaler = fit_scaler(X_train)
X_train, X_val, X_test = apply_scaler(scaler, X_train, X_val, X_test)

# Step 6: Save
save_processed(X_train, y_train, X_val, y_val, X_test, y_test, scaler, config)

Loading raw data from /Users/chris/Documents/higs-vision/data/raw/HIGGS.csv...
Loaded 11000000 events, 28 features
Signal: 5829123, Background: 5170877
Subsampling 1000000 events (stratified)...
Subsampled to 1000000 events
Signal: 529920, Background: 470080
Train: 700000, Val: 150000, Test: 150000
Scaler fitted on training data
  Feature means: [ 9.904e-01 -5.000e-04  5.000e-04]... (first 3)
  Feature stds:  [0.5645 1.0089 1.0068]... (first 3)
Scaler applied to train/val/test
Saved all files to /Users/chris/Documents/higs-vision/data/processed/
  X_test.npy: 33.6 MB
  X_train.npy: 156.8 MB
  X_val.npy: 33.6 MB
  scaler.pkl: 0.0 MB
  y_test.npy: 1.2 MB
  y_train.npy: 5.6 MB
  y_val.npy: 1.2 MB


In [3]:
import sys
sys.path.append("/Users/chris/Documents/higs-vision")

from src.data import load_config
from src.models import HiggsDNN, create_model_from_config

config = load_config("/Users/chris/Documents/higs-vision/config.yaml")
model = create_model_from_config(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nArchitecture:\n{model}")

Total parameters: 189,313
Trainable parameters: 189,313

Architecture:
HiggsDNN(
  (network): Sequential(
    (0): Linear(in_features=28, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=128, out_features=64, bias=True)
    (13): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.3, inplace=False)
    (16): Linear(in_features=64, out_features=1, bias=True)
   

In [4]:
from src.data import load_config, load_processed
from src.models import create_model_from_config
from src.train import train

config = load_config("/Users/chris/Documents/higs-vision/config.yaml")
X_train, y_train, X_val, y_val, X_test, y_test, scaler = load_processed(config)

# Quick test: override max_epochs to 5
config["dnn"]["max_epochs"] = 5
config["dnn"]["early_stopping_patience"] = 3

model = create_model_from_config(config)
model, history = train(model, X_train, y_train, X_val, y_val, config)

Loaded processed data from /Users/chris/Documents/higs-vision/data/processed/
  Train: (700000, 28), Val: (150000, 28), Test: (150000, 28)
Epoch   1 | Train Loss: 0.6107 | Val Loss: 0.5650 | Val AUC: 0.7768 | Time: 16.3s
Epoch   2 | Train Loss: 0.5707 | Val Loss: 0.5422 | Val AUC: 0.7983 | Time: 16.1s
Epoch   3 | Train Loss: 0.5546 | Val Loss: 0.5328 | Val AUC: 0.8066 | Time: 15.9s
Epoch   4 | Train Loss: 0.5476 | Val Loss: 0.5280 | Val AUC: 0.8105 | Time: 15.8s
Epoch   5 | Train Loss: 0.5439 | Val Loss: 0.5263 | Val AUC: 0.8118 | Time: 15.9s

Training complete in 79.9s (1.3 min)
Peak RAM: 0.91 GB
Best validation AUC: 0.8118
